# Feature Selection — Selección de features

**Objetivo:** Encontrar las mejores features usando dos enfoques y comparar resultados.

**Estrategia:**
- **Ruta A:** Partir de tests estadísticos → filtrar → RF importance
- **Ruta B:** Partir de TODAS las features → RF importance directo
- **Comparar:** ¿Coinciden? ¿Cuál funciona mejor?

Esto nos da la base para diseñar los experimentos en YAML.

---
## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
import warnings

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

print('Setup listo.')

---
## 2. Carga de datos

In [ ]:
df = pd.read_csv('../data/processed/application_train_preprocessed.csv')
tests = pd.read_csv('../data/processed/statistical_tests_summary.csv')

TARGET_COL = 'TARGET'
ID_COL = 'SK_ID_CURR'

all_features = [c for c in df.columns if c not in [ID_COL, TARGET_COL]]

print(f'Dataset: {df.shape[0]:,} filas x {df.shape[1]} columnas')
print(f'Total features: {len(all_features)}')
print(f'Tests evaluados: {tests.shape[0]}')

---
## RUTA A — Desde tests estadísticos

Filramos features con evidencia (p < 0.05 Y efecto ≥ 0.1), luego verificamos multicolinealidad, y finalmente evaluamos con Random Forest.

### A.1 Filtrar por evidencia estadística

In [ ]:
util_features = tests[tests['Util_practica'] == True]['Feature'].tolist()

print(f'Features con evidencia: {len(util_features)}')
print()
for f in util_features:
    row = tests[tests['Feature'] == f].iloc[0]
    print(f'  {f:45s} {row["Tipo"]:12s} efecto={row["Effect_size"]:.3f}')

### A.2 Verificar multicolinealidad

In [ ]:
# Matriz de correlación
corr_matrix = df[util_features].corr().abs()

# Encontrar pares altamente correlacionados
high_corr = []
for i in range(len(corr_matrix.columns)):
    for j in range(i+1, len(corr_matrix.columns)):
        if corr_matrix.iloc[i, j] > 0.9:
            high_corr.append({
                'Feature_1': corr_matrix.columns[i],
                'Feature_2': corr_matrix.columns[j],
                'Correlacion': corr_matrix.iloc[i, j]
            })

high_corr_df = pd.DataFrame(high_corr).sort_values('Correlacion', ascending=False)

print(f'Pares con correlación > 0.9: {len(high_corr_df)}')
high_corr_df

In [ ]:
# Eliminar una de cada par
corr_with_target = df.drop(columns=[ID_COL]).corr(method='spearman')[TARGET_COL]

cols_to_remove = set()
for _, row in high_corr_df.iterrows():
    f1, f2 = row['Feature_1'], row['Feature_2']
    if f1 in cols_to_remove or f2 in cols_to_remove:
        continue
    
    corr_f1 = abs(corr_with_target.get(f1, 0))
    corr_f2 = abs(corr_with_target.get(f2, 0))
    
    remove = f2 if corr_f1 >= corr_f2 else f1
    keep = f1 if corr_f1 >= corr_f2 else f2
    
    cols_to_remove.add(remove)
    print(f'  Eliminar: {remove:40s} (corr_target={corr_with_target.get(remove, 0):.4f})')
    print(f'  Mantener: {keep:40s} (corr_target={corr_with_target.get(keep, 0):.4f})')
    print()

route_a_features = [f for f in util_features if f not in cols_to_remove]

print(f'Ruta A después de multicolinealidad: {len(route_a_features)} features')

### A.3 Random Forest en features filtradas

In [ ]:
X_a = df[route_a_features]
y = df[TARGET_COL]

rf_a = RandomForestClassifier(
    n_estimators=200, max_depth=10,
    class_weight='balanced', random_state=42, n_jobs=-1
)
rf_a.fit(X_a, y)

importances_a = pd.Series(rf_a.feature_importances_, index=X_a.columns)
importances_a = importances_a.sort_values(ascending=False)

# Filtrar por importancia mínima
threshold = 0.001
route_a_final = [f for f in importances_a.index if importances_a[f] > threshold]

print('RUTA A — Top 15 por Random Forest')
print('=' * 60)
for i, (col, val) in enumerate(importances_a.head(15).items(), 1):
    marker = '✓' if col in route_a_final else '✗'
    bar = '█' * int(val * 100)
    print(f'  {i:2d}. {marker} {col:40s} {val:.4f} {bar}')

print(f'\nRuta A final: {len(route_a_final)} features')

---
## RUTA B — Desde todas las features

Entrenamos Random Forest con TODAS las features y vemos cuáles son importantes.

### B.1 Random Forest en todas las features

In [ ]:
X_b = df[all_features]
y = df[TARGET_COL]

print(f'Features para Ruta B: {X_b.shape[1]}')
print(f'Muestras: {X_b.shape[0]:,}')

In [ ]:
rf_b = RandomForestClassifier(
    n_estimators=200, max_depth=10,
    class_weight='balanced', random_state=42, n_jobs=-1
)
rf_b.fit(X_b, y)

importances_b = pd.Series(rf_b.feature_importances_, index=X_b.columns)
importances_b = importances_b.sort_values(ascending=False)

# Filtrar por importancia mínima
route_b_final = [f for f in importances_b.index if importances_b[f] > threshold]

print('RUTA B — Top 15 por Random Forest (todas las features)')
print('=' * 60)
for i, (col, val) in enumerate(importances_b.head(15).items(), 1):
    marker = '✓' if col in route_b_final else '✗'
    bar = '█' * int(val * 100)
    print(f'  {i:2d}. {marker} {col:40s} {val:.4f} {bar}')

print(f'\nRuta B final: {len(route_b_final)} features')

---
## COMPARACIÓN — Ambas rutas

¿Coinciden? ¿Cuáles son diferentes?

In [ ]:
# Comparar rutas
set_a = set(route_a_final)
set_b = set(route_b_final)

coinciden = set_a & set_b
solo_a = set_a - set_b
solo_b = set_b - set_a

print('COMPARACIÓN DE RUTAS')
print('=' * 60)
print(f'Ruta A (tests → RF):  {len(route_a_final)} features')
print(f'Ruta B (RF directo):  {len(route_b_final)} features')
print()
print(f'Coinciden:            {len(coinciden)} features')
print(f'Solo en Ruta A:       {len(solo_a)} features')
print(f'Solo en Ruta B:       {len(solo_b)} features')

In [ ]:
# Features que coinciden
print('FEATURES QUE COINCIDEN (ambas rutas)')
print('=' * 60)
for f in sorted(coinciden):
    imp_a = importances_a.get(f, 0)
    imp_b = importances_b.get(f, 0)
    print(f'  {f:40s} RF_A={imp_a:.4f}  RF_B={imp_b:.4f}')

In [ ]:
# Features solo en Ruta A
if solo_a:
    print('FEATURES SOLO EN RUTA A (tests estadísticos)')
    print('=' * 60)
    for f in sorted(solo_a):
        imp_a = importances_a.get(f, 0)
        print(f'  {f:40s} RF_A={imp_a:.4f}')
    print()
    print('Nota: Estas features pasaron los tests pero RF las considera')
    print('menos importantes. Posibrelmente capturan relaciones lineales.')
else:
    print('No hay features solo en Ruta A.')

In [ ]:
# Features solo en Ruta B
if solo_b:
    print('FEATURES SOLO EN RUTA B (RF directo)')
    print('=' * 60)
    for f in sorted(solo_b):
        imp_b = importances_b.get(f, 0)
        print(f'  {f:40s} RF_B={imp_b:.4f}')
    print()
    print('Nota: Estas features no pasaron los tests pero RF las considera')
    print('importantes. Posiblemente capturan relaciones no lineales.')
else:
    print('No hay features solo en Ruta B.')

---
## Diseño de experimentos

Basado en la comparación, podemos diseñar los configs YAML:

In [ ]:
# Crear sets de features para experimentos
feature_sets = {
    'route_a': route_a_final,
    'route_b': route_b_final,
    'intersection': sorted(coinciden),
    'union': sorted(set_a | set_b)
}

print('SETS DE FEATURES PARA EXPERIMENTOS')
print('=' * 60)
for name, features in feature_sets.items():
    print(f'  {name:20s}: {len(features):3d} features')

print()
print('Estos sets se usarán en los archivos YAML de experimentos.')

---
## Guardar resultados

In [ ]:
import os
os.makedirs('../data/processed', exist_ok=True)

# Guardar cada set de features
for name, features in feature_sets.items():
    filepath = f'../data/processed/features_{name}.txt'
    with open(filepath, 'w') as f:
        for feat in features:
            f.write(feat + '\n')
    print(f'Guardado: {filepath} ({len(features)} features)')

# Guardar dataset con la ruta ganadora
df_final = df[[ID_COL, TARGET_COL] + route_a_final]
df_final.to_csv('../data/processed/application_train_selected.csv', index=False)
print(f'\nDataset final (Ruta A): {df_final.shape[0]:,} filas x {df_final.shape[1]} columnas')

---
## Conclusiones para experimentos

### Hallazgos principales:

1. **Ruta A vs Ruta B:** [describir coincidencias y diferencias]
2. **Features exclusivas de Ruta A:** [capturan relaciones lineales]
3. **Features exclusivas de Ruta B:** [capturan relaciones no lineales]
4. **Intersección:** [las más robustas, aparecen en ambos métodos]

### Recomendación para experimentos:

- **exp001:** Ruta A (tests estadísticos)
- **exp002:** Ruta B (RF directo)
- **exp003:** Intersección (las que coinciden)
- **exp004:** Unión (todas las candidatas)

Cada experimento se ejecuta con diferentes modelos (Logistic Regression, Random Forest, etc.) y se compara en MLflow.